### Comp_who_pays

based on impact percent per cell and insurance coverage, computes computes which percentage is payed by who (government, farmer or insurance)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
## Imports

import sys
sys.path.append("..") # Adds the project root to the path

In [10]:
## "F" = farmers, "I" = insurance, "G" = government 

import pandas as pd 

def comp_who_pays(relative_damage, insured): 

    if relative_damage >= 0.5: 

        return pd.Series( 

			[ 

                insured * 0 + (1- insured) * 0.65, 

		        insured * 0.1 + (1- insured) * 0, 

		        insured * 0.9 + (1- insured) * 0.35 

			], 

			index=["F", "I", "G"] 

        ) 

    elif 0.2 <= relative_damage < 0.5 : 

        return pd.Series( 

			[ 

                insured * 0 + (1- insured) * 1, 

                insured * 1 + (1- insured) * 0, 

                insured * 0 + (1- insured) * 0 

            ], 

            index=["F", "I", "G"]	 

        ) 

    else: ## relative_damage < 0.2  

        return pd.Series( 

			[1,0,0], 

            index=["F", "I", "G"] 

        ) 

In [ ]:
## Vectorized Version

import pandas as pd
import numpy as np

def comp_who_pays(relative_damage, insured):
    """_summary_

    Args:
        relative_damage (_type_): _description_
        insured (_type_): _description_

    Returns:
        _type_: _description_
    """

    condlist = [
        relative_damage >= 0.5,
        (relative_damage >= 0.2) & (relative_damage < 0.5),
        relative_damage < 0.2
    ]

    # Payments for conditions in order: F, I, G
    F = np.select(condlist, [
            insured * 0 + (1 - insured) * 0.65,
            insured * 0 + (1 - insured) * 1,
            1
        ])

    I = np.select(condlist, [
            insured * 0.1 + (1 - insured) * 0,
            insured * 1   + (1 - insured) * 0,
            0
        ])

    G = np.select(condlist, [
            insured * 0.9 + (1 - insured) * 0.35,
            0,
            0
        ])

    return pd.DataFrame({"F": F, "I": I, "G": G})

In [8]:
from src.data_exposure import get_exposure
from src.data_hazard import get_haz_dict
from src.data_insurance import get_insurance
from src.helpers import comp_impact

haz_dict = get_haz_dict()
haz_list = list(haz_dict.keys())

exposure = get_exposure(hazard_types=haz_list)

from src.helpers import comp_impact

df = exposure.gdf
df["eai"] = comp_impact(haz_dict=haz_dict,
                        exposure=exposure)

df["insurance_level"] = get_insurance(0.3)

/Users/arvedluetzen/.pyenv/versions/miniforge3-latest/envs/eth-fs2026-praktikum/lib/python3.12/site-packages/climada/hazard/io.py:696: UserWarning: Not all values are of type <class 'str'>. Casting values.
  warnings.warn(


2026-04-02 15:13:12,982 - climada.util.coordinates - WARNING - Distance to closest centroid is greater than 0.078125 degree for 3 coordinates.


/Users/arvedluetzen/.pyenv/versions/miniforge3-latest/envs/eth-fs2026-praktikum/lib/python3.12/site-packages/climada/util/lines_polys_handler.py:638: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  group = gdf_pnts.groupby(axis=0, level=0)


In [23]:
df.join(comp_who_pays(df["eai"], df["insurance_level"]))

,DDEP_C_COD,DDEP_L_LIB,DREG_L_LIB,value,area,impf_WS,geometry,eai,insurance_level,F,I,G
0,01,Ain,Auvergne-Rhône-Alpes,0.199,5762.4,1,"POLYGON ((6.16845 46.36746, 6.16668 46.37074, ...",0.919305,0.3,0.455,0.03,0.515
1,02,Aisne,Hauts-de-France,0.532,7361.7,1,"POLYGON ((4.25573 49.90398, 4.23694 49.90378, ...",1.000000,0.3,0.455,0.03,0.515
2,03,Allier,Auvergne-Rhône-Alpes,0.170,7340.1,1,"POLYGON ((4.00456 46.32748, 3.99436 46.32765, ...",1.000000,0.3,0.455,0.03,0.515
3,04,Alpes-de-Haute-Provence,Provence-Alpes-Côte d'Azur,0.025,6925.2,1,"POLYGON ((6.96709 44.62287, 6.9539 44.63783, 6...",0.808035,0.3,0.455,0.03,0.515
4,05,Hautes-Alpes,Provence-Alpes-Côte d'Azur,0.017,5548.7,1,"POLYGON ((7.07587 44.68512, 7.07424 44.69209, ...",0.570663,0.3,0.455,0.03,0.515
...,...,...,...,...,...,...,...,...,...,...,...,...
91,91,Essonne,Île-de-France,0.403,1804.4,1,"POLYGON ((2.58407 48.67715, 2.58038 48.68948, ...",1.000000,0.3,0.455,0.03,0.515
92,92,Hauts-de-Seine,Île-de-France,NaN,175.6,1,"POLYGON ((2.33598 48.93158, 2.33491 48.94154, ...",1.000000,0.3,0.455,0.03,0.515
93,93,Seine-Saint-Denis,Île-de-France,NaN,236.2,1,"POLYGON ((2.6026 48.92936, 2.6024 48.93532, 2....",1.000000,0.3,0.455,0.03,0.515
94,94,Val-de-Marne,Île-de-France,0.029,245.0,1,"POLYGON ((2.61482 48.76112, 2.60645 48.77333, ...",1.000000,0.3,0.455,0.03,0.515


In [24]:
comp_who_pays(
    np.linspace(0, 1, 11),
    np.linspace(0, 1, 11)
)

,F,I,G
0,1.000,0.00,0.000
1,1.000,0.00,0.000
2,0.800,0.20,0.000
3,0.700,0.30,0.000
4,0.600,0.40,0.000
5,0.325,0.05,0.625
6,0.260,0.06,0.680
7,0.195,0.07,0.735
8,0.130,0.08,0.790
9,0.065,0.09,0.845
